<a href="https://colab.research.google.com/github/nakamura196/NDLOCR-GoogleColabVersion/blob/main/NDLOCRv2_googlecolabversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
CONTENT_DIR = str(Path("/content"))

!pip install pip==24.0
!pip install numpy==1.26.4
!pip install --trusted-host download.pytorch.org torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 torchtext==0.16.1 --index-url https://download.pytorch.org/whl/cu121

%cd {CONTENT_DIR}

!git clone --recursive https://github.com/ndl-lab/ndlocr_cli -b feature/colab
# #2. 必要なパッケージをインストールする
PROJECT_DIR=str(Path(f"{CONTENT_DIR}/ndlocr_cli"))

!pip install hydra-colorlog
!pip install hydra-core
!pip install mmpretrain==1.2.0
!pip install pytorch-lightning==1.6.5
!pip install datasets

!pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html

!pip install torchmetrics==0.11.4

!pip uninstall -y torchao
!pip install transformers==4.45.2

%cd {PROJECT_DIR}/submodules/ndl_layout
!git clone https://github.com/open-mmlab/mmdetection.git -b v3.3.0
%cd {PROJECT_DIR}/submodules/ndl_layout/mmdetection
#下行はGPUのメモリ不足になった場合にコメントアウトを外すとよい。
!sed -i -e 's/GPU_MEM_LIMIT = 1024\*\*3/GPU_MEM_LIMIT = 1024\*\*3\/\/5/' mmdet/models/roi_heads/mask_heads/fcn_mask_head.py
!python setup.py bdist_wheel
!pip install dist/*.whl

%cd {PROJECT_DIR}
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/resnet-orient2.ckpt -P ./submodules/text_recognition_lightning/models
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/rf_author/model.pkl -P ./submodules/text_recognition_lightning/models/rf_author/
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/text_recognition_lightning/rf_title/model.pkl -P ./submodules/text_recognition_lightning/models/rf_title/
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/ndl_layout/ndl_retrainmodel.pth -P ./submodules/ndl_layout/models
!wget -nc https://lab.ndl.go.jp/dataset/ndlocr_v2/separate_pages_mmdet/epoch_180.pth -P ./submodules/separate_pages_mmdet/models

# OCRの実行

In [14]:
%cd {PROJECT_DIR}
!mkdir -p input/img

!wget -O input/img/sample.jpg https://dl.ndl.go.jp/api/iiif/3437686/R0000001/full/1200,/0/default.jpg

%cd {PROJECT_DIR}
!python main.py infer input output -s s -x